<a href="https://colab.research.google.com/github/Anshulx007/AI-ML/blob/main/Customer_Segmentation_CaseStudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")

print(df.head())
print("Dataset shape:", df.shape)
print(df.info())

FileNotFoundError: [Errno 2] No such file or directory: 'marketing_campaign.csv'

In [ ]:

df["Income"] = df["Income"].fillna(df["Income"].median())

df["Dt_Customer"] = pd.to_datetime(
    df["Dt_Customer"],
    dayfirst=True,
    errors="coerce"
)

latest_date = df["Dt_Customer"].max()

df["Customer_Tenure_Days"] = (
    latest_date - df["Dt_Customer"]
).dt.days

df.drop(
    columns=[
        "ID",
        "Dt_Customer",
        "Z_CostContact",
        "Z_Revenue",
        "Response"
    ],
    inplace=True,
    errors="ignore"
)

print(df.head())

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include="object").columns

df[num_cols] = df[num_cols].fillna(
    df[num_cols].median()
)

for col in cat_cols:
    df[col] = df[col].fillna(
        df[col].mode()[0]
    )

df_encoded = pd.get_dummies(
    df,
    columns=cat_cols,
    dtype=int
)

print(df_encoded.head())
print("Encoded shape:", df_encoded.shape)

In [ ]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df_encoded["Total_Spending"] = df_encoded[
    spending_cols
].sum(axis=1)

purchase_cols = [
    "NumDealsPurchases",
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases"
]

df_encoded["Total_Purchases"] = df_encoded[
    purchase_cols
].sum(axis=1)

df_encoded.drop(
    columns=spending_cols + purchase_cols,
    inplace=True,
    errors="ignore"
)

print(df_encoded.head())

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(df_encoded)

print("Scaled data shape:", X_scaled.shape)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)

print("Original features:", X_scaled.shape[1])
print("Features after PCA:", X_pca.shape[1])

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_pca)

    score = silhouette_score(
        X_pca,
        labels,
        sample_size=1000,
        random_state=42
    )

    print(f"K = {k}, Silhouette Score = {score:.4f}")

In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled_new)

print(df["KMeans_Cluster"].value_counts())

df["Total_Spending"] = df[
    [
        "MntWines",
        "MntFruits",
        "MntMeatProducts",
        "MntFishProducts",
        "MntSweetProducts",
        "MntGoldProds"
    ]
].sum(axis=1)

summary = df.groupby("KMeans_Cluster")[
    ["Income", "Total_Spending"]
].mean().round(2)

print(summary)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_new)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=df["KMeans_Cluster"],
    palette="viridis",
    s=40
)

plt.title("Customer Segmentation using K-Means and PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.show()

In [ ]:
print("Number of customers in each cluster:")

print(df["KMeans_Cluster"].value_counts())

spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_cols].sum(axis=1)

summary = df.groupby("KMeans_Cluster")[
    ["Income", "Total_Spending"]
].mean().round(2)

print("\nCustomer Segmentation Summary:")
print(summary)

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled_new)

print("DBSCAN Cluster Labels:")
print(set(dbscan_labels))

print("\nNumber of customers in each cluster:")
print(pd.Series(dbscan_labels).value_counts().sort_index())

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_new)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=dbscan_labels,
    palette="viridis",
    s=40
)

plt.title("Customer Segmentation using DBSCAN and PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    max_iter=1000
)

X_tsne = tsne.fit_transform(X_scaled_new)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    hue=df["KMeans_Cluster"],
    palette="viridis",
    s=40
)

plt.title("Customer Segmentation using K-Means and t-SNE")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score
import pandas as pd

kmeans_score = silhouette_score(
    X_scaled_new,
    df["KMeans_Cluster"],
    sample_size=1000,
    random_state=42
)

mask = dbscan_labels != -1
dbscan_valid_labels = dbscan_labels[mask]
dbscan_valid_data = X_scaled_new[mask]

if len(set(dbscan_valid_labels)) >= 2:
    dbscan_score = silhouette_score(
        dbscan_valid_data,
        dbscan_valid_labels,
        sample_size=min(1000, len(dbscan_valid_labels)),
        random_state=42
    )
else:
    dbscan_score = None

comparison = pd.DataFrame({
    "Algorithm": ["K-Means", "DBSCAN"],
    "Silhouette Score": [kmeans_score, dbscan_score]
})

print(comparison)

In [ ]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_cols].sum(axis=1)

cluster_summary = df.groupby("KMeans_Cluster").agg(
    Customer_Count=("KMeans_Cluster", "count"),
    Average_Income=("Income", "mean"),
    Average_Spending=("Total_Spending", "mean"),
    Average_Web_Purchases=("NumWebPurchases", "mean"),
    Average_Store_Purchases=("NumStorePurchases", "mean")
).round(2)

print("Customer Segmentation Analysis:")
print(cluster_summary)